## 1. Imports and Data

In [63]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf


In [64]:

tickers = {
    "Information Technology": ["AAPL", "MSFT", "NVDA", "AVGO", "CRM"],  # High momentum, growth factor exposure
    "Financials": ["JPM", "BAC", "GS", "MS", "BLK", "WFC"],             # Value factor, rate sensitivity
    "Health Care": ["JNJ", "UNH", "LLY", "ABBV", "MRK", "PFE"],         # Defensive, quality factor
    "Consumer Discretionary": ["AMZN", "TSLA", "HD", "MCD", "NKE", "LOW"], # Cyclical, momentum variation
    "Industrials": ["CAT", "HON", "UNP", "RTX", "GE", "DE"],            # Classic value/quality mix
    "Communication Services": ["GOOGL", "META", "DIS", "NFLX", "T"],    # Growth vs. value spread
    "Consumer Staples": ["PG", "KO", "PEP", "WMT", "COST", "CL"],       # Low vol, defensive
    "Energy": ["XOM", "CVX", "COP", "SLB", "EOG"],                      # Value, commodity beta
    "Utilities": ["NEE", "DUK", "SO", "AEP", "EXC"],                    # Low vol, yield factor
    "Real Estate": ["PLD", "AMT", "EQIX", "SPG", "PSA"],                # Yield, rate sensitivity
    "Materials": ["LIN", "APD", "NEM", "FCX", "SHW"],                   # Cyclical, commodity exposure
}

all_tickers = [ticker for sector in tickers.values() for ticker in sector]



In [66]:
start_date = "2015-01-01"
end_date = "2026-08-01"

data = yf.download(
    [ticker for sector in tickers.values() for ticker in sector],
    start=start_date,
    end=end_date,
    interval = "1mo"
)["Close"]

# Compute returns
returns = np.log(data).diff().dropna()

# Data Quality Checks
# 1. Are there any NaNs?
print(returns.isna().sum().sum())

# 2. Are they at the start?
print(returns.iloc[0].isna().sum())

# 3. Flag outliers (returns > 50% or < -50%)
outliers = (returns > 0.5) | (returns < -0.5)
outliers_df = returns[outliers].stack(level = 'Ticker')
print(outliers_df)


C:\Users\godwi\AppData\Local\Temp\ipykernel_19688\556127669.py:4: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(
[*********************100%***********************]  60 of 60 completed

0
0
Date        Ticker
2016-02-01  FCX       0.506032
2020-03-01  EOG      -0.565959
            SLB      -0.697216
            SPG      -0.808050
2020-08-01  TSLA      0.554719
2022-04-01  NFLX     -0.676915
dtype: float64


In [67]:
# Compare to benchmark
spy_row = yf.download(
    "SPY",
    start=start_date,
    end=end_date,
    interval = "1mo"
)

spy_prices = spy_row["Close"]["SPY"]

spy_returns = np.log(spy_prices).diff().dropna()
spy_returns = spy_returns.reindex(returns.index)
#spy_returns.to_csv("data/spy_returns.csv")

# Check shape
print(returns.shape)        # (T, N)
print(spy_returns.shape)    # (T,)
# Confirm all share the same index
assert returns.index.equals(spy_returns.index)

assert len(returns) == len(spy_returns), \
    "returns and spy_returns have different lengths"


C:\Users\godwi\AppData\Local\Temp\ipykernel_19688\3775808830.py:2: FutureWarning: YF.download() has changed argument auto_adjust default to True
  spy_row = yf.download(
[*********************100%***********************]  1 of 1 completed

(138, 60)
(138,)


## 2. Raw Signals

In [68]:
# Pull Fundamental Data
# Market Cap
shares_dict = {}
for ticker in all_tickers:
    try:
        info = yf.Ticker(ticker).info
        shares = info.get('sharesOutstanding', None)
        shares_dict[ticker] = shares
    except Exception as e:
        print(f"Error fetching data for {ticker}: {e}")
        shares_dict[ticker] = None

shares_series = pd.Series(shares_dict)
missing = shares_series[shares_series.isna()]
print(missing) # None
market_cap = data.multiply(shares_series, axis=1)
#market_cap.to_csv("data/market_cap.csv")
log_market_cap = np.log(market_cap)
#log_market_cap.to_csv("data/log_market_cap.csv")

print(log_market_cap.shape) # 120 Months x 60 stocks
print(log_market_cap.head())

Series([], dtype: int64)
(139, 60)
                 AAPL       ABBV        AEP        AMT       AMZN        APD  \
Date                                                                           
2015-01-01  26.665470  24.935654  23.848913  24.249752  25.973893  23.843573   
2015-02-01  26.761485  24.938136  23.770497  24.272089  26.043693  23.913394   
2015-03-01  26.729611  24.905203  23.747128  24.220445  26.022263  23.887090   
2015-04-01  26.735381  25.012896  23.758090  24.228818  26.147584  23.833805   
2015-05-01  26.779722  25.042307  23.757365  24.210240  26.165093  23.856757   

                 AVGO        BAC        BLK        CAT  ...        SLB  \
Date                                                    ...              
2015-01-01  24.329738  25.163941  24.409739  24.051857  ...  25.222113   
2015-02-01  24.545231  25.206584  24.496628  24.087841  ...  25.249189   
2015-03-01  24.543152  25.182781  24.487342  24.052608  ...  25.240597   
2015-04-01  24.460270  25.217267  

In [69]:
from statsmodels.regression.linear_model import OLS
from statsmodels.tools import add_constant
# Compute Raw Signal Values
# 1. Market Beta
# Run OLS over past 36 months
beta_window = 36
monthly_index = returns.index
beta_panel = pd.DataFrame(np.nan, index = monthly_index, columns = all_tickers)

assert returns.index.equals(spy_returns.index), \
    "returns and spy_returns have different indices — align them first"

for i, date in enumerate(monthly_index):
    if i < beta_window:
        continue  # Not enough data for the first few months


    window_returns = returns.iloc[i-beta_window:i]
    window_spy = spy_returns.iloc[i-beta_window:i]
    
    valid_mask = window_spy.notna() 
    if valid_mask.sum() < 24:
        continue

    spy_window_clean = window_spy[valid_mask] 

    for ticker in all_tickers:
        y = window_returns.loc[window_spy.index, ticker]

        combined_valid = valid_mask & y.notna()

        if combined_valid.sum() < 24: 
            continue

        y_clean = y[combined_valid].values
        spy_clean = window_spy[combined_valid].values
        X_clean = add_constant(spy_clean, has_constant="add")  # Recreate X with cleaned spy data

        try:
            model = OLS(y_clean, X_clean).fit()
            beta_panel.loc[date, ticker] = model.params[1]  # Store the beta coefficient

        except Exception:
            pass

#beta_panel.to_csv("data/signal_beta_raw.csv")


    

In [70]:
# Sanity check
print((beta_panel < -1).sum().sum())
print((beta_panel > 3).sum().sum()) 
beta_panel.describe()

0
0


,AAPL,MSFT,NVDA,AVGO,CRM,JPM,BAC,GS,MS,BLK,...,PLD,AMT,EQIX,SPG,PSA,LIN,APD,NEM,FCX,SHW
count,102.000000,102.000000,102.000000,102.000000,102.000000,102.000000,102.000000,102.000000,102.000000,102.000000,...,102.000000,102.000000,102.000000,102.000000,102.000000,102.000000,102.000000,102.000000,102.000000,102.000000
mean,1.161925,0.966485,1.912940,1.023582,1.196982,1.149022,1.452174,1.357544,1.371910,1.357635,...,1.093189,0.605712,0.711855,1.410551,0.490676,0.793268,0.804417,0.263555,1.900906,1.205211
std,0.119308,0.132395,0.502833,0.255817,0.163113,0.089500,0.166697,0.140509,0.197999,0.168764,...,0.290870,0.355845,0.325753,0.551330,0.422560,0.216009,0.116560,0.275695,0.413200,0.152263
min,0.816838,0.749326,0.923618,0.298525,0.870039,0.933790,1.162995,0.993032,0.939029,1.067052,...,0.691077,-0.034936,0.150482,0.418341,-0.147801,0.372995,0.354439,-0.401131,0.880386,0.949859
25%,1.103242,0.853809,1.422021,0.932109,1.073142,1.080802,1.338493,1.263679,1.196452,1.210536,...,0.830958,0.241751,0.377031,0.926800,0.101398,0.614336,0.740351,0.164065,1.641572,1.064514
50%,1.192165,0.988182,2.000444,1.026681,1.166422,1.154869,1.462305,1.340643,1.398400,1.393849,...,1.022836,0.714063,0.674268,1.430090,0.375094,0.793444,0.799998,0.348007,1.863051,1.194686
75%,1.244682,1.034478,2.420430,1.198305,1.329614,1.212008,1.602508,1.465999,1.532379,1.485763,...,1.373715,0.865543,1.063516,1.938940,0.982256,0.986499,0.869320,0.453704,2.206611,1.301984
max,1.353873,1.314641,2.727052,1.615780,1.602712,1.421825,1.728807,1.669259,1.725444,1.701663,...,1.733342,1.129348,1.233570,2.233657,1.132597,1.160070,1.054834,0.736323,2.830854,1.563775


In [71]:
"""
fig, axes = plt.subplots(10, 6, figsize=(18, 28))
axes = axes.flatten()

for i, ticker in enumerate(beta_panel.columns):
    ax = axes[i]
    beta_panel[ticker].plot(ax=ax, linewidth=1.5)
    mean_beta = beta_panel[ticker].mean()
    ax.axhline(mean_beta, color="red", ls="--", linewidth=1, label="Mean")
    ax.set_title(f"{ticker}, Mean Beta = {mean_beta:.2f}")
    ax.set_ylabel("Beta")

for j in range(len(beta_panel.columns), len(axes)):
    fig.delaxes(axes[j])

fig.suptitle("Rolling 36-Month Beta vs SPY", y=1.002)
fig.tight_layout()
plt.savefig("outputs/beta_panel.png")
plt.show()
"""

'\nfig, axes = plt.subplots(10, 6, figsize=(18, 28))\naxes = axes.flatten()\n\nfor i, ticker in enumerate(beta_panel.columns):\n    ax = axes[i]\n    beta_panel[ticker].plot(ax=ax, linewidth=1.5)\n    mean_beta = beta_panel[ticker].mean()\n    ax.axhline(mean_beta, color="red", ls="--", linewidth=1, label="Mean")\n    ax.set_title(f"{ticker}, Mean Beta = {mean_beta:.2f}")\n    ax.set_ylabel("Beta")\n\nfor j in range(len(beta_panel.columns), len(axes)):\n    fig.delaxes(axes[j])\n\nfig.suptitle("Rolling 36-Month Beta vs SPY", y=1.002)\nfig.tight_layout()\nplt.savefig("outputs/beta_panel.png")\nplt.show()\n'

In [72]:
# 2. Momentum
mom_start = 12 # 12 months lookback
mom_end = 1 # exclude most recent month - noisy
momentum_panel = pd.DataFrame(np.nan, index = monthly_index, columns = all_tickers)

for i, date in enumerate(monthly_index):
    if i < mom_start:
        continue  # Not enough data for the first few months

    window = returns.iloc[i-mom_start:i-mom_end] 
    compounded = (1 + window).prod(axis = 0, skipna = False) - 1
    momentum_panel.loc[date] = compounded

#momentum_panel.to_csv("data/signal_momentum_raw.csv")
print(momentum_panel)

                AAPL      MSFT      NVDA      AVGO       CRM       JPM  \
Date                                                                     
2015-02-01       NaN       NaN       NaN       NaN       NaN       NaN   
2015-03-01       NaN       NaN       NaN       NaN       NaN       NaN   
2015-04-01       NaN       NaN       NaN       NaN       NaN       NaN   
2015-05-01       NaN       NaN       NaN       NaN       NaN       NaN   
2015-06-01       NaN       NaN       NaN       NaN       NaN       NaN   
...              ...       ...       ...       ...       ...       ...   
2026-03-01  0.054916  0.058279  0.432182  0.536899 -0.325147  0.164223   
2026-04-01  0.174706  0.021190  0.542687  0.788864 -0.311413  0.236644   
2026-05-01  0.179865 -0.086373  0.510778  0.523209 -0.342622  0.207036   
2026-06-01  0.332167 -0.131911  0.410347  0.609748 -0.369813  0.197700   
2026-07-01  0.487948 -0.112726  0.288374  0.518290 -0.338873  0.045385   

                 BAC        GS       

In [73]:
# 3. Size
size_panel = log_market_cap.copy()
size_panel = size_panel.reindex(index=monthly_index)
size_panel = size_panel.reindex(columns=all_tickers)

assert size_panel.index.equals(monthly_index), \
    "Index mismatch between size panel and returns index"

assert size_panel.columns.tolist() == all_tickers, \
    "Column mismatch between size panel and returns columns"

#size_panel.to_csv("data/signal_size_raw.csv")

In [74]:
# 4. Low Volatility
vol_window = 12
low_vol_panel = -returns.rolling(window = vol_window, min_periods = 10).std() # Negative for low vol = high score
low_vol_panel = low_vol_panel.reindex(index=monthly_index)
low_vol_panel = low_vol_panel.reindex(columns=all_tickers)
#low_vol_panel.to_csv("data/signal_lowvol_raw.csv")

In [75]:
# 5. Long-term Reversal
ltr_start = 36
ltr_end = 12
ltr_panel = pd.DataFrame(np.nan, index = monthly_index, columns = all_tickers)

for i, date in enumerate(monthly_index):
    if i < ltr_start:
        continue  
    
    window = returns.iloc[i-ltr_start:i-ltr_end] 
    compounded = (1 + window).prod(axis = 0, skipna = False) - 1
    ltr_panel.loc[date] = -compounded 

#ltr_panel.to_csv("data/signal_ltr_raw.csv")

In [76]:
'''
dates = np.load("data/dates.npy", allow_pickle=True) 
tickers = np.load("data/tickers.npy", allow_pickle=True)  
factors = np.load("data/factors.npy", allow_pickle=True) 


data = pd.read_csv("data/prices.csv", index_col=0, parse_dates=True)
returns = pd.read_csv("data/returns.csv", index_col=0, parse_dates=True).reindex(dates)
spy_returns = pd.read_csv("data/spy_returns.csv", index_col=0, parse_dates=True).reindex(dates)
mktcap_panel = pd.read_csv("data/market_cap.csv", index_col=0, parse_dates=True).reindex(dates)
beta_panel = pd.read_csv("data/signal_beta_raw.csv", index_col=0, parse_dates=True)
momentum_panel = pd.read_csv("data/signal_momentum_raw.csv", index_col=0, parse_dates=True)
size_panel = pd.read_csv("data/signal_size_raw.csv", index_col=0, parse_dates=True)
low_vol_panel = pd.read_csv("data/signal_lowvol_raw.csv", index_col=0, parse_dates=True)
ltr_panel = pd.read_csv("data/signal_ltr_raw.csv", index_col=0, parse_dates=True)
'''



print(beta_panel.shape)
print(momentum_panel.shape)
print(size_panel.shape)
print(low_vol_panel.shape)
print(ltr_panel.shape)

(138, 60)
(138, 60)
(138, 60)
(138, 60)
(138, 60)


In [77]:
beta_panel.tail()

,AAPL,MSFT,NVDA,AVGO,CRM,JPM,BAC,GS,MS,BLK,...,PLD,AMT,EQIX,SPG,PSA,LIN,APD,NEM,FCX,SHW
Date,,,,,,,,,,,,,,,,,,,,,
2026-03-01,0.895635,1.148675,2.127050,1.201821,1.276640,1.195227,1.416526,1.382584,1.503591,1.393850,...,1.606177,0.713017,0.946208,1.304083,1.026583,0.588942,0.740185,0.196887,0.960315,1.492749
2026-04-01,0.848416,1.082689,1.973904,1.191313,1.125964,1.214758,1.430974,1.361881,1.470121,1.453150,...,1.565420,0.798484,0.844469,1.381007,1.107614,0.578276,0.585842,0.460835,1.132484,1.563775
2026-05-01,0.816838,1.076419,1.825467,1.452237,0.870039,1.109888,1.348760,1.262925,1.479785,1.393848,...,1.449976,0.798920,0.873078,1.296217,1.132597,0.493722,0.561875,0.395305,0.919049,1.320092
2026-06-01,0.894654,1.132802,1.855187,1.484035,0.939555,1.027923,1.265246,1.266670,1.461379,1.333540,...,1.419508,0.771362,0.845625,1.234268,1.097555,0.458378,0.473906,0.318457,0.949524,1.236898
2026-07-01,0.897508,1.245749,1.901492,1.615780,1.121205,0.978400,1.239499,1.366860,1.502316,1.384895,...,1.506363,0.808357,0.852101,1.144369,1.068163,0.388661,0.354439,0.409735,0.880386,1.069003


In [93]:
# Winsorise, Z-score normalisation and Burn-in
# Set values < 1%ile to 1%ile, > 99%ile to 99%ile, across all stocks, at each time t - hence 'cross sectional'
def winsorise(signal_panel: pd.DataFrame, lower = 0.01, upper = 0.99) -> pd.DataFrame:
    def winsorise_row(row):
        lower_bound = row.quantile(lower)
        upper_bound = row.quantile(upper)
        return row.clip(lower=lower_bound, upper=upper_bound)
    
    return signal_panel.apply(winsorise_row, axis=1)

def standardise(signal_panel: pd.DataFrame) -> pd.DataFrame:
    def standardise_row(row):
        mu = row.mean()
        sig = row.std()
        if sig == 0:
            return row - mu
        return (row - mu) / sig

    return signal_panel.apply(standardise_row, axis=1)

def trim_burn_in(signal_panel: pd.DataFrame, burn_in_periods = 36) -> pd.DataFrame:
    return signal_panel.iloc[burn_in_periods:]

def normalise(df: pd.DataFrame) -> pd.DataFrame:
    def normalise_row(row):
        tot = row.abs().sum()
        return row / tot

    return df.apply(normalise_row, axis=1)



In [ ]:
signals_raw = {
    "beta": beta_panel,
    "momentum": momentum_panel,
    "size": size_panel,
    "low_vol": low_vol_panel,
    "ltr": ltr_panel
}

signals_processed = {
    name: trim_burn_in(winsorise(standardise(panel))) for name, panel in signals_raw.items() 
}

#np.save("data/signals_raw.npy", signals_raw)
#np.save("data/signals_processed.npy", signals_processed)

In [80]:
# Trim signals due to burn-in
for signal in signals_processed.values():
    print(signal.isna().sum().sum())
    print(signal.shape)

# No NaNs


0
(102, 60)
0
(102, 60)
0
(102, 60)
0
(102, 60)
0
(102, 60)


In [81]:
signals_processed["beta"].tail()

,AAPL,MSFT,NVDA,AVGO,CRM,JPM,BAC,GS,MS,BLK,...,PLD,AMT,EQIX,SPG,PSA,LIN,APD,NEM,FCX,SHW
Date,,,,,,,,,,,,,,,,,,,,,
2026-03-01,0.118249,0.577899,2.134774,0.674439,0.810347,0.662461,1.064451,1.002795,1.222606,1.023261,...,1.408955,-0.213477,0.210117,0.860199,0.356117,-0.438861,-0.164126,-1.151032,0.235741,1.202911
2026-04-01,0.006823,0.431885,1.970050,0.628972,0.510402,0.671511,1.063810,0.938448,1.134837,1.104047,...,1.307748,-0.083772,-0.000338,0.973151,0.477109,-0.483315,-0.469589,-0.696400,0.522233,1.304762
2026-05-01,0.094319,0.567947,1.931201,1.253658,0.191390,0.629014,1.064856,0.908244,1.303923,1.147124,...,1.249534,0.061628,0.196934,0.968988,0.670449,-0.495233,-0.370882,-0.674803,0.280812,1.012549
2026-06-01,0.295042,0.722266,1.880137,1.352358,0.375592,0.534118,0.959864,0.962418,1.311716,1.082379,...,1.236601,0.073863,0.207087,0.904290,0.659034,-0.487614,-0.459756,-0.738623,0.393475,0.909008
2026-07-01,0.312108,0.927348,1.788042,1.581086,0.707315,0.455021,0.916306,1.141318,1.380629,1.173179,...,1.387778,0.154604,0.231887,0.748240,0.613606,-0.586877,-0.647338,-0.549645,0.281858,0.615090


## 3. Orthogonalise and weights via Information Coefficient

In [82]:
def residualise(target: pd.DataFrame, controls):
    if isinstance(controls, (pd.DataFrame, pd.Series)):
        controls = [controls]
    elif isinstance(controls, tuple):
        controls = list(controls)
    elif not isinstance(controls, list):
        raise TypeError("controls must be a DataFrame, Series, list, or tuple")

    controls = [
        ctrl.reindex(index=target.index, columns=target.columns)
        if isinstance(ctrl, pd.DataFrame)
        else ctrl.reindex(index=target.index)
        for ctrl in controls
    ]

    residuals = pd.DataFrame(index=target.index, columns=target.columns, dtype=float)

    for t in target.index:
        y = target.loc[t]
        X = pd.concat(
            [ctrl.loc[t].rename(f"ctrl_{i}") for i, ctrl in enumerate(controls)],
            axis=1,
        )

        common_idx = y.index.intersection(X.index)
        y = y.reindex(common_idx)
        X = X.reindex(common_idx)

        valid = y.notna() & X.notna().all(axis=1)
        if valid.sum() < 2:
            continue

        y_clean = y.loc[valid]
        X_clean = X.loc[valid]
        X_clean = add_constant(X_clean, has_constant="add")

        try:
            model = OLS(y_clean, X_clean).fit()
            residuals.loc[t, X_clean.index] = model.resid
        except Exception:
            continue

    return residuals

# 1. Size orthogonal to beta
size_clean = residualise(signals_processed["size"], signals_processed["beta"])

# 2. low vol orthogonal to beta + size
lowvol_clean = residualise(
    signals_processed["low_vol"],
    [signals_processed["beta"], size_clean],
)

# 3. momentum orthogonal to beta + size + low vol
momentum_clean = residualise(
    signals_processed["momentum"],
    [signals_processed["beta"], size_clean, lowvol_clean],
)

# 4. ltr orthogonal to beta + size + low vol + momentum
ltr_clean = residualise(
    signals_processed["ltr"],
    [signals_processed["beta"], size_clean, lowvol_clean, momentum_clean],
)

signals_cleaned = {
    "beta": signals_processed["beta"],
    "momentum": momentum_clean,
    "size": size_clean,
    "low_vol": lowvol_clean,
    "ltr": ltr_clean
}
    
#np.save("data/signals_cleaned.npy", signals_cleaned)

dates = signals_cleaned["beta"].index
factors = list(signals_cleaned.keys())




In [83]:
print(signals_processed["beta"].index[-1])
print(signals_cleaned["beta"].index[-1])
print(dates[-1])
print(id(dates))
print(id(signals_cleaned["beta"].index))

2026-07-01 00:00:00
2026-07-01 00:00:00
2026-07-01 00:00:00
1976131543616
1976131543616


In [84]:
returns = returns.reindex(dates)

def compute_ic(name: str, signal: pd.DataFrame, returns: pd.DataFrame):
    returns = returns.shift(-1) # align r_{t+1}
    ic_series = pd.DataFrame(index = signal.index, columns = [name])
    
    for t in signal.index:
        x = signal.loc[t]
        r = returns.loc[t]

        valid = x.notna() & r.notna()

        ic = x[valid].corr(r[valid])
        ic_series.loc[t] = ic

    return ic_series



In [ ]:
ic_df = pd.DataFrame(index = dates)
for name, signal in signals_processed.items():
    ic_df = pd.concat([ic_df, compute_ic(name, signal, returns)], axis = 1)
ic_df.dropna(inplace = True)
weights_rolling = standardise(ic_df).rolling(window = 24).mean().dropna()
weights_rolling

,beta,momentum,size,low_vol,ltr
Date,,,,,
2020-01-01,-0.024434,-0.080284,0.137946,0.061537,-0.094765
2020-02-01,-0.031783,-0.025736,0.203471,0.038915,-0.184868
2020-03-01,0.046389,-0.031480,0.142511,-0.008786,-0.148634
2020-04-01,0.006005,-0.011632,0.120721,0.050705,-0.165799
2020-05-01,0.086911,0.044849,0.098383,-0.008947,-0.221196
...,...,...,...,...,...
2026-02-01,0.026573,0.040612,0.042498,-0.179624,0.069941
2026-03-01,0.118239,0.075456,0.044644,-0.271824,0.033486
2026-04-01,0.102525,0.081651,0.077853,-0.269193,0.007164


In [86]:
t_eval = weights_rolling.index
#np.save('data/t_eval.npy', t_eval)

for name, signal in signals_cleaned.items():
    signals_cleaned[name] = signal.reindex(t_eval)

signal_matrix = np.stack(
    [
        signals_cleaned["beta"].values,   
        signals_cleaned["momentum"].values,     
        signals_cleaned["size"].values,    
        signals_cleaned["low_vol"].values,   
        signals_cleaned["ltr"].values  
    ],
    axis=2   # stack along the 3rd dimension
)

signal_matrix.shape

(78, 60, 5)

In [87]:
weights = weights_rolling.to_numpy()
scores = np.sum(signal_matrix * weights[:, None, :], axis = 2)
scores = pd.DataFrame(
    scores, 
    index = t_eval, 
    columns = all_tickers
)

#scores.to_csv('data/factor_scores.csv')
scores

,AAPL,MSFT,NVDA,AVGO,CRM,JPM,BAC,GS,MS,BLK,...,PLD,AMT,EQIX,SPG,PSA,LIN,APD,NEM,FCX,SHW
Date,,,,,,,,,,,,,,,,,,,,,
2020-01-01,0.164067,0.379589,-0.174256,0.010834,0.282976,0.065379,0.005400,-0.217425,-0.147960,-0.117418,...,-0.081402,0.017882,-0.217745,-0.081633,-0.209556,0.018877,-0.174370,-0.254070,-0.416753,-0.068947
2020-02-01,0.382780,0.552694,-0.032350,0.137342,0.381035,0.142917,0.079879,-0.338243,-0.203005,-0.130487,...,0.037689,0.267746,-0.251075,-0.308981,-0.260581,0.065755,-0.134084,-0.250152,-0.580182,-0.083488
2020-03-01,0.276057,0.493543,0.192381,0.092019,0.356987,0.126947,0.128703,-0.249000,-0.138656,-0.029112,...,-0.000952,0.023196,-0.207481,-0.120467,-0.371787,0.030450,-0.096140,-0.304113,-0.220079,-0.049412
2020-04-01,0.205671,0.459843,0.180843,0.090762,0.344340,0.080671,0.092066,-0.213451,-0.096017,-0.069184,...,0.038986,0.087629,-0.208142,-0.008829,-0.346358,0.064070,0.018075,-0.259313,-0.179230,0.007460
2020-05-01,0.198470,0.472142,0.224908,0.091054,0.379568,0.119914,0.161672,-0.178971,-0.012339,-0.028258,...,0.004383,0.040603,-0.286314,0.317929,-0.366285,0.009569,0.072055,-0.195933,-0.003718,0.014085
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-02-01,-0.029996,0.096672,0.143445,0.321580,0.111367,-0.183555,-0.057961,0.077791,0.017935,-0.192228,...,-0.064725,-0.155515,0.078653,-0.120699,-0.147502,-0.105530,-0.073726,0.111558,0.281061,-0.201175
2026-03-01,-0.166709,0.121681,0.427691,0.459033,0.213144,-0.305367,-0.141924,0.034489,-0.077473,-0.080846,...,-0.080398,-0.268739,0.039847,-0.163715,-0.087918,-0.204808,-0.124151,0.498880,0.583684,-0.063310
2026-04-01,-0.160959,0.086261,0.417594,0.698779,0.122328,-0.275407,-0.184997,0.010659,-0.027737,-0.090745,...,-0.212518,-0.243736,0.091145,-0.157055,-0.023362,-0.184768,-0.177275,0.547481,0.513523,-0.117069


## 4. Evaluate Portfolio

In [89]:
returns_eval = returns.shift(-1).reindex(t_eval)
#returns_eval.to_csv('data/returns_eval.csv')
returns_eval.tail()

Ticker,AAPL,ABBV,AEP,AMT,AMZN,APD,AVGO,BAC,BLK,CAT,...,SLB,SO,SPG,T,TSLA,UNH,UNP,WFC,WMT,XOM
Date,,,,,,,,,,,,,,,,,,,,,
2026-02-01,-0.040124,-0.064929,-0.020688,-0.105905,-0.008272,0.052365,-0.029828,-0.016275,-0.094802,-0.047374,...,0.000973,-0.008871,-0.077318,0.034389,-0.079498,-0.072745,-0.088158,-0.022850,-0.027052,0.106632
2026-03-01,0.066902,-0.020517,0.044976,0.066981,0.241121,0.038630,0.299126,0.092228,0.102580,0.230153,...,0.101500,0.001863,0.088105,-0.093455,0.026230,0.314195,0.104998,0.032381,0.059734,-0.094703
2026-04-01,0.140707,0.029836,-0.071962,0.022995,0.020833,-0.074100,0.067927,-0.035412,-0.017724,-0.016127,...,-0.041826,-0.041036,0.005873,-0.052240,0.132709,0.026187,-0.020528,-0.053025,-0.128932,-0.053803
2026-05-01,-0.075524,0.144790,0.077008,-0.124153,-0.127091,0.050938,-0.166230,0.099179,-0.085013,0.195419,...,-0.159880,0.038991,0.098422,-0.180710,-0.035478,0.094504,0.035018,0.063700,-0.021747,-0.060590
2026-06-01,0.088767,-0.006899,-0.021873,0.008704,0.035852,0.015259,0.060003,0.039065,0.058687,-0.126471,...,0.016004,-0.005658,-0.017774,0.016292,-0.033975,0.037889,0.046827,0.050379,-0.009314,0.005398


In [91]:
from sklearn.decomposition import PCA

lookback = 36
window = returns.iloc[-lookback-1:-1]
pca = PCA()
pca.fit(window)
variance_pct = pca.explained_variance_ratio_
cum_var = np.cumsum(variance_pct)
k = np.argmax(cum_var >= 0.8) + 1

In [92]:
lookback = 36
threshold = 0.8
sigma_list = []

for t in range(lookback, len(returns)):
    # Lookback window
    window = returns.iloc[t-lookback:t]
    print(returns.index[t])
    # PCA deconstruction
    pca = PCA()
    pca.fit(window)

    cum_var = np.cumsum(pca.explained_variance_ratio_)
    k = np.argmax(cum_var >= threshold)

    B = pca.components_[:k,:]
    F = np.diag(pca.explained_variance_[:k])

    sigma_pca = B.T @ F @ B

    sample_cov = window.cov().values

    D = np.diag(np.diag(sample_cov - sigma_pca))

    sigma_t = sigma_pca + D

    sigma_list.append(sigma_t)

# Stack along axis 2
sigma = np.stack(sigma_list, axis=2)
sigma


2021-02-01 00:00:00
2021-03-01 00:00:00
2021-04-01 00:00:00
2021-05-01 00:00:00
2021-06-01 00:00:00
2021-07-01 00:00:00
2021-08-01 00:00:00
2021-09-01 00:00:00
2021-10-01 00:00:00
2021-11-01 00:00:00
2021-12-01 00:00:00
2022-01-01 00:00:00
2022-02-01 00:00:00
2022-03-01 00:00:00
2022-04-01 00:00:00
2022-05-01 00:00:00
2022-06-01 00:00:00
2022-07-01 00:00:00
2022-08-01 00:00:00
2022-09-01 00:00:00
2022-10-01 00:00:00
2022-11-01 00:00:00
2022-12-01 00:00:00
2023-01-01 00:00:00
2023-02-01 00:00:00
2023-03-01 00:00:00
2023-04-01 00:00:00
2023-05-01 00:00:00
2023-06-01 00:00:00
2023-07-01 00:00:00
2023-08-01 00:00:00
2023-09-01 00:00:00
2023-10-01 00:00:00
2023-11-01 00:00:00
2023-12-01 00:00:00
2024-01-01 00:00:00
2024-02-01 00:00:00
2024-03-01 00:00:00
2024-04-01 00:00:00
2024-05-01 00:00:00
2024-06-01 00:00:00
2024-07-01 00:00:00
2024-08-01 00:00:00
2024-09-01 00:00:00
2024-10-01 00:00:00
2024-11-01 00:00:00
2024-12-01 00:00:00
2025-01-01 00:00:00
2025-02-01 00:00:00
2025-03-01 00:00:00


array([[[ 9.35940105e-03,  9.69532440e-03,  9.48055033e-03, ...,
          3.34649923e-03,  3.77348400e-03,  3.83491038e-03],
        [ 1.87154360e-03,  1.72688842e-03,  1.20303558e-03, ...,
          6.27981676e-04,  8.71403506e-04,  3.71560147e-04],
        [ 1.34407104e-04,  5.16142056e-04,  4.41138557e-04, ...,
          7.37592161e-04,  5.81565118e-04,  4.84543706e-05],
        ...,
        [ 3.48397928e-03,  2.85737880e-03,  2.56520000e-03, ...,
          4.49896849e-04,  3.70874823e-04,  2.80237581e-05],
        [ 1.36974243e-03,  1.50840654e-03,  1.41731780e-03, ...,
          6.26877973e-04,  1.35675961e-04,  2.84488431e-04],
        [ 4.12728266e-03,  3.47945048e-03,  3.50263120e-03, ...,
         -1.28418171e-03, -1.30595709e-03, -1.12640127e-03]],

       [[ 1.87154360e-03,  1.72688842e-03,  1.20303558e-03, ...,
          6.27981676e-04,  8.71403506e-04,  3.71560147e-04],
        [ 8.04031611e-03,  8.08121036e-03,  6.85691785e-03, ...,
          3.81849601e-03,  3.51013243e

In [96]:
scores_n = normalise(scores)

scores_n

,AAPL,MSFT,NVDA,AVGO,CRM,JPM,BAC,GS,MS,BLK,...,PLD,AMT,EQIX,SPG,PSA,LIN,APD,NEM,FCX,SHW
Date,,,,,,,,,,,,,,,,,,,,,
2020-01-01,0.020171,0.046668,-0.021423,0.001332,0.034790,0.008038,0.000664,-0.026731,-0.018191,-0.014436,...,-0.010008,0.002198,-0.026770,-0.010036,-0.025763,0.002321,-0.021437,-0.031236,-0.051237,-0.008476
2020-02-01,0.032573,0.047032,-0.002753,0.011687,0.032425,0.012162,0.006797,-0.028783,-0.017275,-0.011104,...,0.003207,0.022784,-0.021366,-0.026293,-0.022174,0.005596,-0.011410,-0.021287,-0.049371,-0.007105
2020-03-01,0.032443,0.058002,0.022609,0.010814,0.041954,0.014919,0.015126,-0.029263,-0.016295,-0.003421,...,-0.000112,0.002726,-0.024384,-0.014158,-0.043693,0.003579,-0.011299,-0.035740,-0.025864,-0.005807
2020-04-01,0.026271,0.058737,0.023100,0.011593,0.043983,0.010304,0.011760,-0.027265,-0.012264,-0.008837,...,0.004980,0.011193,-0.026586,-0.001128,-0.044241,0.008184,0.002309,-0.033123,-0.022893,0.000953
2020-05-01,0.021529,0.051215,0.024396,0.009877,0.041173,0.013007,0.017537,-0.019414,-0.001338,-0.003065,...,0.000475,0.004404,-0.031057,0.034487,-0.039732,0.001038,0.007816,-0.021253,-0.000403,0.001528
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-02-01,-0.003517,0.011335,0.016818,0.037704,0.013057,-0.021521,-0.006796,0.009121,0.002103,-0.022538,...,-0.007589,-0.018234,0.009222,-0.014152,-0.017294,-0.012373,-0.008644,0.013080,0.032953,-0.023587
2026-03-01,-0.011814,0.008623,0.030308,0.032529,0.015104,-0.021640,-0.010057,0.002444,-0.005490,-0.005729,...,-0.005697,-0.019044,0.002824,-0.011602,-0.006230,-0.014514,-0.008798,0.035353,0.041362,-0.004486
2026-04-01,-0.012106,0.006488,0.031407,0.052554,0.009200,-0.020713,-0.013913,0.000802,-0.002086,-0.006825,...,-0.015983,-0.018331,0.006855,-0.011812,-0.001757,-0.013896,-0.013333,0.041175,0.038621,-0.008805


In [ ]:
sigma_t = sigma[:,:,-len(t_eval):]
weights_t = pd.DataFrame(
    index = t_eval, 
    columns = scores.columns
)
for i,t in enumerate(t_eval):
    mu_t = scores_n.loc[t].values
    sigma_t_i = sigma[:,:,i]

    w = np.linalg.solve(sigma_t_i, mu_t)
    w = w - w.mean() # denorm
    w = w / np.abs(w).sum() # scale exposure

    weights_t.loc[t] = w

print(weights_t.sum(axis = 1))
print(weights_t.abs().sum(axis = 1))